# 23 - VLM-named prompts: does the USS region help Qwen + SAM3 find unseen obstacles?

Notebook 22: the USS region contains the obstacle 76-97% of the time, but SAM3
with the generic word "object" returns no candidates on most non-cone frames,
so the region could not be used. Here Qwen2.5-VL-7B NAMES the obstacle (and
boxes it), and SAM3 segments with that noun. Asked twice per frame, with the
same prompt:

- **crop** = the USS-region crop (USS tells Qwen where to look) -> `uss_*`
- **full** = the whole frame (no USS) -> `nouss_*`

So the only difference between the two arms is whether the USS chose the view.
Same frames, calibration and GT as notebook 22 (sequences are a subset of its
sample). Two resumable stages (Qwen, then SAM3) so the models never share the
GPU. **GPU: L4 or A100** (Qwen 7B in bf16; same as notebook 18).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    # canmatrix 1.3.0 (2026-10-01) breaks `import asammdf` (TypeError in
    # asammdf/blocks/types.py) - pin until asammdf supports it
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf==8.8.27",
                    "canmatrix==1.2"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import json as _json
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_DIR / "scripts"))
import uss_prompt_analysis as upa
from vision_uss_research.labeling.corridor import pick_target_instance
from vision_uss_research.method.uss_prompt import (fit_ground_camera, region_box,
                                                   track_target_slots)
from vision_uss_research.method.vlm_prompt import (name_prompt, padded_crop,
                                                   parse_name_box)
from vision_uss_research.qa.flags import box_iou
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import frames_window_id

DATASET_RUN = "20260718_dataset_v2"
TYPES_RUN = "dataset_v2daysplit"          # manifest carrying target_object per sequence
PROJ_RUN = "prompt_uss_proj_v1"
TYPES = ["cone", "bush", "bollard", "car", "pole", "curbstone"]
UNSEEN = {"cone", "bush", "bollard", "car"}
SEQS_PER_TYPE = 40          # <= 60; sequences are a subset of notebook 22's sample
FRAMES_PER_SEQ = 5
REGION = dict(half_w=160, below=100, above=250)
QWEN_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
SCORE_THRESHOLD = 0.2
FOUND_IOU = 0.3
MIN_GT_H, MIN_GT_W = 15, 8
SEED = 0

MAN_DIR = ARTIFACTS / "datasets" / DATASET_RUN
PROJ_DIR = ARTIFACTS / "method" / PROJ_RUN
WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
RUN_DIR = ARTIFACTS / "method" / new_run_id("prompt", "unseen_vlm_v1")
QWEN_CSV = RUN_DIR / "qwen.csv"
SAM_CSV = RUN_DIR / "sam3.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ["sequence_id", "camera", "frame_idx"]

slots, boxes = upa.load(PROJ_DIR, MAN_DIR)
types = (pd.read_csv(ARTIFACTS / "datasets" / TYPES_RUN / "dataset_manifest.csv",
                     usecols=["sequence_id", "target_object"]).drop_duplicates("sequence_id"))
pairs = (slots.merge(track_target_slots(slots), on=["sequence_id", "camera", "slot"])
         .merge(boxes, on=KEYS).merge(types, on="sequence_id"))
pairs["day"] = pairs.sequence_id.str.split("_").str[1]
pairs = pairs[((pairs.y1 - pairs.y0) >= MIN_GT_H) & ((pairs.x1 - pairs.x0) >= MIN_GT_W)]

# cross-fit calibration: two folds per (camera, day), global per fold as fallback
seq_fold = {}
for (_, _), d in pairs.groupby(["camera", "day"]):
    for i, s in enumerate(sorted(d.sequence_id.unique())):
        seq_fold[s] = i % 2
pairs["fold"] = pairs.sequence_id.map(seq_fold)
def fit(d, cam_name):
    return fit_ground_camera(d.mid_x, d.mid_y, (d.x0 + d.x1) / 2, d.y1, cam_name)[0]
glob = {(c, f): fit(pairs[(pairs.camera == c) & (pairs.fold == f)], c)
        for c in ("front", "rear") for f in (0, 1)}
cams = {}
for (c, day, f), d in pairs.groupby(["camera", "day", "fold"]):
    if len(d) >= 25:
        cams[(c, day, f)] = fit(d, c)
print(f"per-day-fold calibrations: {len(cams)}")

rng = np.random.default_rng(SEED)
chosen = []
for t in TYPES:
    seqs = pairs[pairs.target_object == t].sequence_id.drop_duplicates()
    seqs = seqs.sample(min(60, len(seqs)), random_state=SEED).head(SEQS_PER_TYPE)
    sub = pairs[pairs.sequence_id.isin(seqs)].sample(frac=1, random_state=SEED)
    chosen.append(sub.groupby("sequence_id").head(FRAMES_PER_SEQ))
todo = pd.concat(chosen, ignore_index=True)
proj = []
for r in todo.itertuples(index=False):
    other = 1 - r.fold
    m = cams.get((r.camera, r.day, other))
    src = "day"
    if m is None:
        m, src = glob[(r.camera, other)], "global"
    u, v, ok = m.project(r.mid_x, r.mid_y)
    proj.append((float(u), float(v), bool(ok), src))
todo["pu"], todo["pv"], todo["proj_ok"], todo["calib"] = zip(*proj)
todo = todo[todo.proj_ok]

todo["item"] = todo.sequence_id + "|" + todo.camera + "|" + todo.frame_idx.astype(str)
start_run(RUN_DIR, config={"types": TYPES, "seqs_per_type": SEQS_PER_TYPE, "qwen": QWEN_MODEL,
                           "region": REGION, "found_iou": FOUND_IOU},
          inputs=[f"method/{PROJ_RUN}", f"datasets/{DATASET_RUN}", f"datasets/{TYPES_RUN}"],
          repo_root=REPO_DIR)
print(todo.groupby("target_object").agg(seqs=("sequence_id", "nunique"),
                                         frames=("frame_idx", "size")).to_string())
print(f"calibration: {todo.calib.value_counts().to_dict()} | todo frames {len(todo)}")

## Stage 1 - Qwen

In [ ]:
# ---- stage 1: Qwen names + boxes the obstacle, on the USS crop and on the full frame
import subprocess
import torch

def run_stage(path, fn, every=25):
    done = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["item"])
    rest = todo[~todo.item.isin(set(done["item"]))]
    buf = []
    for k, r in enumerate(tqdm(rest.itertuples(index=False), total=len(rest), unit="frame"), 1):
        buf.append({"item": r.item, **fn(r)})
        if len(buf) >= every or k == len(rest):
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
            done.to_csv(path, index=False); buf = []
    return pd.read_csv(path)

if not (QWEN_CSV.exists() and len(pd.read_csv(QWEN_CSV)) >= len(todo)):
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qwen-vl-utils"], check=True)
    from qwen_vl_utils import process_vision_info
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    qmodel = Qwen2_5_VLForConditionalGeneration.from_pretrained(QWEN_MODEL, torch_dtype=dtype,
                                                                device_map="auto")
    qproc = AutoProcessor.from_pretrained(QWEN_MODEL)

    def ask(pil, view):
        msgs = [{"role": "user", "content": [{"type": "image", "image": pil},
                                             {"type": "text", "text": name_prompt(view)}]}]
        text = qproc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(msgs)
        inputs = qproc(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(qmodel.device)
        _, gh, gw = inputs["image_grid_thw"][0].tolist()
        with torch.inference_mode():
            out = qmodel.generate(**inputs, max_new_tokens=80, do_sample=False)
        reply = qproc.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)[0]
        name, box = parse_name_box(reply, (gw * 14, gh * 14), pil.size)
        return name, box, reply.strip()[:200]

    def qwen_one(r):
        p = WINDOW_ROOT / r.sequence_id / r.camera / f"{int(r.frame_idx):05d}.jpg"
        if not p.exists():
            return {"error": "no_frame"}
        img = Image.open(p).convert("RGB")
        W, H = img.size
        reg = region_box(r.pu, r.pv, W, H, **REGION)
        cx0, cy0, cx1, cy1 = padded_crop(reg, W, H)
        row = {"error": "", "crop": f"{cx0},{cy0},{cx1},{cy1}"}
        for view, pil, off in (("crop", img.crop((cx0, cy0, cx1, cy1)), (cx0, cy0)),
                               ("full", img, (0, 0))):
            name, box, reply = ask(pil, view)
            row[f"{view}_name"] = name
            row[f"{view}_reply"] = reply
            if box is not None:
                box = box + np.array([off[0], off[1], off[0], off[1]])
            for c, v in zip(("x0", "y0", "x1", "y1"),
                            box if box is not None else [np.nan] * 4):
                row[f"{view}_q{c}"] = float(v)
        return row

    qwen = run_stage(QWEN_CSV, qwen_one)
    del qmodel
    torch.cuda.empty_cache()
qwen = pd.read_csv(QWEN_CSV)
print(f"Qwen rows {len(qwen)} | crop named {qwen.crop_name.notna().mean():.0%} | "
      f"full named {qwen.full_name.notna().mean():.0%}")

## Stage 2 - SAM3

In [ ]:
# ---- stage 2: SAM3 segments with each Qwen noun; keep the mask that matches
from vision_uss_research.labeling.sam3 import Sam3Engine
eng = Sam3Engine(score_threshold=SCORE_THRESHOLD)
qwen_by_item = qwen.set_index("item")

def inside(b, reg):
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return reg[0] <= cx <= reg[2] and reg[1] <= cy <= reg[3]

def qbox(q, view):
    v = [q[f"{view}_q{c}"] for c in ("x0", "y0", "x1", "y1")]
    return None if any(pd.isna(x) for x in v) else tuple(map(float, v))

def sam_one(r):
    q = qwen_by_item.loc[r.item]
    if q.get("error", "") == "no_frame" or str(q.get("error")) == "no_frame":
        return {"error": "no_frame"}
    p = WINDOW_ROOT / r.sequence_id / r.camera / f"{int(r.frame_idx):05d}.jpg"
    img = np.array(Image.open(p).convert("RGB"))
    H, W = img.shape[:2]
    reg = region_box(r.pu, r.pv, W, H, **REGION)
    cache, row = {}, {"error": ""}
    for view in ("crop", "full"):
        name = q[f"{view}_name"]
        pick = None
        if isinstance(name, str) and name:
            if name not in cache:
                res = eng.segment_text(img, name)
                cache[name] = [(tuple(map(float, b)), float(s))
                               for b, s in zip(res["boxes"], res["scores"])]
            cand = cache[name]
            ref = qbox(q, view)
            if view == "crop":                 # USS: keep masks inside the region
                cand = [c for c in cand if inside(c[0], reg)]
            if cand:
                if ref is not None:
                    pick = max(cand, key=lambda c: box_iou(c[0], ref))
                elif view == "crop":
                    pick = max(cand, key=lambda c: c[1])
                else:
                    i, _ = pick_target_instance({"boxes": np.array([c[0] for c in cand]),
                                                 "scores": np.array([c[1] for c in cand])}, (H, W))
                    pick = cand[i] if i is not None else None
            row[f"{view}_n_cand"] = len(cache.get(name, []))
        row[f"{view}_sam_box"] = None if pick is None else ",".join(f"{v:.0f}" for v in pick[0])
    eng.free()
    return row

sam = run_stage(SAM_CSV, sam_one)

## Summary

In [ ]:
res = (todo.merge(qwen, on="item", suffixes=("", "_q")).merge(sam, on="item", suffixes=("", "_s")))
res = res[res.error.fillna("") == ""]
res["unseen"] = res.target_object.isin(UNSEEN).astype(int)
def parse(s):
    return None if not isinstance(s, str) else tuple(map(float, s.split(",")))
gt = list(zip(res.x0, res.y0, res.x1, res.y1))
VARIANTS = {
    "uss_qwen_box":   [None if pd.isna(a) else (a, b, c, d) for a, b, c, d in
                       zip(res.crop_qx0, res.crop_qy0, res.crop_qx1, res.crop_qy1)],
    "uss_qwen_sam":   [parse(s) for s in res.crop_sam_box],
    "nouss_qwen_box": [None if pd.isna(a) else (a, b, c, d) for a, b, c, d in
                       zip(res.full_qx0, res.full_qy0, res.full_qx1, res.full_qy1)],
    "nouss_qwen_sam": [parse(s) for s in res.full_sam_box],
}
for k, boxes_ in VARIANTS.items():
    res[f"{k}_iou"] = [box_iou(b, g) if b is not None else 0.0 for b, g in zip(boxes_, gt)]
rows = []
for t, d in res.groupby("target_object"):
    row = {"type": t, "unseen": int(t in UNSEEN), "seqs": d.sequence_id.nunique(), "frames": len(d)}
    for k in VARIANTS:
        f = d[f"{k}_iou"] >= FOUND_IOU
        row[f"{k}_frame"] = round(f.mean(), 3)
        row[f"{k}_seq"] = round(d.assign(f=f).groupby("sequence_id").f.any().mean(), 3)
    rows.append(row)
tab = pd.DataFrame(rows).sort_values(["unseen", "type"], ascending=[False, True])
pd.set_option("display.width", 260); display(tab)
tab.to_csv(RUN_DIR / "summary.csv", index=False)

u = res[res.unseen == 1]
for a_, b_ in (("uss_qwen_sam", "nouss_qwen_sam"), ("uss_qwen_box", "nouss_qwen_box")):
    a = u[f"{a_}_iou"] >= FOUND_IOU; b = u[f"{b_}_iou"] >= FOUND_IOU
    print(f"UNSEEN frames {len(u)}: {a_} {a.mean():.3f} vs {b_} {b.mean():.3f} | "
          f"USS-only wins {int((a & ~b).sum())}, no-USS-only wins {int((~a & b).sum())}")
print("\nwhat Qwen calls each type (crop view, top 4):")
for t, d in res.groupby("target_object"):
    print(f"  {t:10s} {d.crop_name.value_counts().head(4).to_dict()}")
res.to_csv(RUN_DIR / "scored.csv", index=False)
finish_run(RUN_DIR, summary={"rows": tab.to_dict(orient="records")})

import matplotlib.pyplot as plt
g = res[res.target_object.isin(["bush", "bollard", "car"])].sample(min(12, len(res)), random_state=SEED)
fig, axes = plt.subplots(3, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, g.itertuples()):
    im = cv2.cvtColor(cv2.imread(str(WINDOW_ROOT / r.sequence_id / r.camera /
                                     f"{int(r.frame_idx):05d}.jpg")), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    reg = region_box(r.pu, r.pv, W, H, **REGION)
    cv2.rectangle(im, (int(reg[0]), int(reg[1])), (int(reg[2]), int(reg[3])), (255, 220, 0), 2)
    cv2.rectangle(im, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), (0, 255, 0), 2)
    for b, color in ((parse(r.crop_sam_box), (255, 0, 0)), (parse(r.full_sam_box), (0, 120, 255))):
        if b:
            cv2.rectangle(im, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), color, 2)
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{r.target_object}: crop '{r.crop_name}' {r.uss_qwen_sam_iou:.2f} / "
                 f"full '{r.full_name}' {r.nouss_qwen_sam_iou:.2f}", fontsize=7)
fig.tight_layout(); fig.savefig(RUN_DIR / "gallery_vlm.jpg", dpi=90); plt.show()
print(f"download {RUN_DIR.name}/ and share summary.csv + gallery_vlm.jpg + the printout above")